# Process GenX CEM Results

In [ ]:
from ipywidgets import Dropdown, SelectMultiple
from upath import UPath
from src import runner
from ipyfilechooser import FileChooser
import xlwings as xw
from loguru import logger
from tqdm.notebook import trange, tqdm

In [ ]:
genx_wb = FileChooser(default_path=".", default_filename="Kentucky Load Resource Model.xlsb", title="Connect to a GenX spreadsheet: ", filter_pattern="*.xls*", show_hidden=False)
genx_wb

In [ ]:
selected_cases = SelectMultiple(
    options=runner.get_solved_cases(UPath("./cases")),
    description="Availabe CEM cases: ",
    rows=10,
    layout=dict(width="max-content"),
    style=dict(description_width="max-content"),
)
selected_cases

In [ ]:
logger.info(f"Opening {genx_wb.value} in new Excel instance")

if UPath(genx_wb.value).parts[-1] in xw.books:
    xw.books[UPath(genx_wb.value).parts[-1]].save()

with xw.App(visible=False) as xw_sandbox:
    results_wb = xw.apps[xw_sandbox.pid].books.open(genx_wb.value)
    
    cases_to_run = [c for c in results_wb.sheets["Batch Cases"].range("CasesToRun").options(empty=None).value if c is not None]
    
    results_wb.screen_updating = False    
    results_wb.app.calculate()
    
    for case_folder in tqdm(selected_cases.value, desc="Processing results"):
        try:
            runner.load_case_results(
                wb=results_wb,
                report_wb=xw.Book("./Compiled Results.xlsm"),
                base_folder=UPath(case_folder),
                save_view=True,
            )
        except Exception:
            logger.error(f"Oops! Skipping {case_folder}.")